# Run Model

Single-country simulation runner with standard macro, benchmark, permanent-income, sector, and firm-credit diagnostics. Configure the inputs, run the workflow once, then inspect the manifest and diagnostic sections below.

## Companion notebooks

- `run_model_exploration.ipynb` — exploratory firm, household, reader, and ratio analysis
- `run_sensitivity.ipynb` — parameter sensitivity
- `run_mpc.ipynb` — household MPC experiment
- `run_irf.ipynb` — macro impulse responses
- `run_model_legacy_2026-08-18.ipynb` — preserved historical notebook; do not use for new work

In [20]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from src.monte_carlo import run_seeded_monte_carlo
from config import (
    BALANCE_SHEET_COLUMNS,
    FIGURE_SIZES,
    FISCAL_COLUMNS,
    LABOUR_COLUMNS,
    MACRO_COLUMNS,
    POLICY_COLUMNS,
    SCENARIO_PRESETS,
)
from src.notebook_state import run_notebook_workflow, validate_notebook_state
from src.notebook_workflow import (
    NotebookRunConfig,
    build_permanent_income_forecast_contribution_table,
    build_permanent_income_log_ratio_decomposition_df,
    plot_permanent_income_log_ratio_decomposition,
)
from src.visual_helpers import (
    firm_sector_groups_table,
    plot_agent_timeseries,
    plot_cumulative_insolvent_firms_by_sector,
    plot_firm_credit_to_equity_and_capital,
    plot_mc,
    plot_output,
    plot_sector_tfp_investment_desired_mb_mc_ratio,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Inputs

All execution switches are centralized here. Sensitivity, MPC, and IRF switches are in their dedicated notebooks.

In [21]:
RUN_BENCHMARK = True
RUN_MONTE_CARLO = False
SCENARIO_NAME = "calibrated_consumption"
# SCENARIO_NAME = "change_sectoral_weights"

run_config = NotebookRunConfig(
    seed=69,
    t_max=50,
    country_iso3="FRA",
    run_benchmark=RUN_BENCHMARK,
    force_rebuild_data=True,
    force_rerun_benchmark=True,
    benchmark_overrides=None,
)
scenario_overrides = SCENARIO_PRESETS[SCENARIO_NAME]

## Run simulation

In [22]:
# In the first run, state is none. After, run_notebook_workflow uses prepared from the previous state, preventing reloading it.
state = globals().get("state")

state = run_notebook_workflow(
    run_config,
    scenario_name=SCENARIO_NAME,
    scenario_overrides=scenario_overrides,
    previous_state=state,
)

# Familiar aliases; `state` remains the authoritative carrier.
COUNTRY = state.country_code
prepared = state.prepared
data = prepared.data
cfg = prepared.cfg
simulation = state.simulation
model = state.model
df_scenario = state.df_scenario
benchmark = state.benchmark
df_benchmark = state.df_benchmark

validate_notebook_state(state)

Configuration summary
{'productivity_growth': 'SimpleTFPGrowth',
 'productivity_investment_planner': 'TargetIntensityTFPInvestmentPlanner',
 'labour_market': {'name': 'DefaultLabourMarketClearer',
                   'parameters': {'allow_switching_industries': True,
                                  'compare_with_normalised_inputs': True,
                                  'consider_reservation_wages': True,
                                  'firing_cost_fraction': 0.0,
                                  'firing_speed': 1.0,
                                  'hiring_cost_fraction': 0.0,
                                  'hiring_speed': 0.75,
                                  'individuals_quitting': False,
                                  'individuals_quitting_temperature': 1.0,
                                  'optimised_hiring': True,
                                  'random_firing_probability': 0.0,
                                  'round_target_employment': True,
                 

In [23]:
{
    "country": COUNTRY,
    "seed": cfg.seed,
    "t_max": cfg.t_max,
    "scenario": state.scenario_name,
    "model_h5": str(simulation.model_h5_path),
    "benchmark": benchmark is not None,
    "manifest": str(state.manifest_path),
}


{'country': 'FRA',
 'seed': 69,
 't_max': 50,
 'scenario': 'calibrated_consumption',
 'model_h5': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/simulation_FRA.h5',
 'benchmark': True,
 'manifest': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/run_manifest_FRA_seed69_t50.json'}

## Macro, fiscal, policy, labour, and balance sheets

In [24]:
plot_output(
    df=df_scenario[list(MACRO_COLUMNS)],
    no_rows=5,
    no_cols=4,
    country_code=COUNTRY,
    line_color="#1f77b4",
    **FIGURE_SIZES["benchmark"],
)
plot_output(df=df_scenario[list(FISCAL_COLUMNS)], no_rows=5, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_scenario[list(POLICY_COLUMNS)], no_rows=4, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_scenario[list(LABOUR_COLUMNS)], no_rows=2, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_scenario[list(BALANCE_SHEET_COLUMNS)], no_rows=3, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])


## Benchmark comparison

In [25]:
if df_benchmark is None:
    print("Benchmark disabled in Inputs.")
else:
    plot_output(df=df_scenario[list(MACRO_COLUMNS)], df_compare=df_benchmark[list(MACRO_COLUMNS)], no_rows=5, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
    plot_output(df=df_scenario[list(FISCAL_COLUMNS)], df_compare=df_benchmark[list(FISCAL_COLUMNS)], no_rows=5, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
    plot_output(df=df_scenario[list(POLICY_COLUMNS)], df_compare=df_benchmark[list(POLICY_COLUMNS)], no_rows=3, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
    plot_output(df=df_scenario[list(BALANCE_SHEET_COLUMNS)], df_compare=df_benchmark[list(BALANCE_SHEET_COLUMNS)], no_rows=3, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])


## Fiscal diagnostics

In [26]:
def fiscal_diagnostic_table(df, items, periods=(1, 20, 50), unit=1e9, currency="€"):
    """Amount (bn) and % of GDP for each fiscal line item, at the given periods."""
    rows = {}
    for label, col in items.items():
        ratio_col = f"{col}_to_gdp"
        row = {}
        for t in periods:
            amount = df.loc[t, col] / unit
            pct = df.loc[t, ratio_col] * 100 if ratio_col in df.columns else float("nan")
            row[f"Period {t}"] = f"{currency}{amount:,.3f}bn ({pct:.2f}%)"
        rows[label] = row
    return pd.DataFrame(rows).T


REVENUE_ITEMS = {
    "Corporate income tax": "fiscal_revenue_corporate_income_taxes",
    "Household income tax": "fiscal_revenue_income_taxes",
    "Production tax": "fiscal_revenue_production_taxes",
    "VAT": "fiscal_revenue_vat",
    "Capital-formation taxes": "fiscal_revenue_capital_formation_taxes",
    "Export taxes": "fiscal_revenue_export_taxes",
    "Employee social insurance": "fiscal_revenue_employee_social_insurance",
    "Employer social insurance": "fiscal_revenue_employer_social_insurance",
    "Total revenue": "fiscal_revenue",
}

EXPENDITURE_ITEMS = {
    "Government consumption": "government_consumption",
    "Unemployment benefits": "unemployment_benefits",
    "Interest payments on debt": "interest_payments_on_debt",
    "Household social transfers": "household_social_transfers",
    "Public pension benefits": "public_pension_benefits",
    "Other social transfers": "other_social_transfers",
    "Necessity support": "necessity_support",
    "Total expenditure": "fiscal_expenditure",
}


from IPython.display import Markdown

fiscal_periods = (1, 20, 50)
revenue_table = fiscal_diagnostic_table(df_scenario, REVENUE_ITEMS, periods=fiscal_periods)
expenditure_table = fiscal_diagnostic_table(df_scenario, EXPENDITURE_ITEMS, periods=fiscal_periods)
print("Scenario tables")
display(Markdown("**Revenue**\n\n" + revenue_table.to_markdown()))
display(Markdown("**Expenditure**\n\n" + expenditure_table.to_markdown()))


REVENUES_COLS = [
    "debt_to_gdp",
    "deficit_to_gdp",
    "fiscal_revenue_to_gdp",
    "fiscal_expenditure_to_gdp",

    "fiscal_revenue_vat_to_gdp",
    "fiscal_revenue_production_taxes_to_gdp",
    "fiscal_revenue_capital_formation_taxes_to_gdp",
    "fiscal_revenue_corporate_income_taxes_to_gdp",
    "fiscal_revenue_income_taxes_to_gdp",
    "fiscal_revenue_rental_income_taxes_to_gdp",
    "fiscal_revenue_employee_social_insurance_to_gdp",
    "fiscal_revenue_employer_social_insurance_to_gdp",
    "fiscal_revenue_taxes_on_products_to_gdp",
    "fiscal_revenue_export_taxes_to_gdp",
    "fiscal_revenue_social_housing_rent_to_gdp",

    "fiscal_revenue_vat",
    "fiscal_revenue_production_taxes",
    "fiscal_revenue_capital_formation_taxes",
    "fiscal_revenue_corporate_income_taxes",
    "fiscal_revenue_income_taxes",
    "fiscal_revenue_rental_income_taxes",
    "fiscal_revenue_employee_social_insurance",
    "fiscal_revenue_employer_social_insurance",
    "fiscal_revenue_taxes_on_products",
    "fiscal_revenue_export_taxes",
    "fiscal_revenue_social_housing_rent",
    ]

EXPENDITURE_COLS = [
    "debt_to_gdp",
    "deficit_to_gdp",
    "fiscal_revenue_to_gdp",
    "fiscal_expenditure_to_gdp",

    "government_consumption_to_gdp",
    "unemployment_benefits_to_gdp",
    "interest_payments_on_debt_to_gdp",
    "household_social_transfers_to_gdp",
    "public_pension_benefits_to_gdp",
    "other_social_transfers_to_gdp",
    "necessity_support_to_gdp",

    "government_consumption",
    "unemployment_benefits",
    "interest_payments_on_debt",
    "household_social_transfers",
    "public_pension_benefits",
    "other_social_transfers",
    "necessity_support",
]

plot_output(df=df_scenario[list(REVENUES_COLS)], no_rows=7, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_scenario[list(EXPENDITURE_COLS)], no_rows=6, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])


Scenario tables


**Revenue**

|                           | Period 1            | Period 20           | Period 50           |
|:--------------------------|:--------------------|:--------------------|:--------------------|
| Corporate income tax      | €32.438bn (6.29%)   | €32.279bn (5.26%)   | €51.944bn (6.96%)   |
| Household income tax      | €43.649bn (8.46%)   | €48.929bn (7.98%)   | €52.739bn (7.07%)   |
| Production tax            | €8.279bn (1.61%)    | €9.385bn (1.53%)    | €11.293bn (1.51%)   |
| VAT                       | €31.993bn (6.20%)   | €42.179bn (6.88%)   | €54.507bn (7.30%)   |
| Capital-formation taxes   | €4.773bn (0.93%)    | €22.775bn (3.71%)   | €25.493bn (3.42%)   |
| Export taxes              | €0.000bn (0.00%)    | €0.000bn (0.00%)    | €0.000bn (0.00%)    |
| Employee social insurance | €30.265bn (5.87%)   | €33.347bn (5.44%)   | €35.503bn (4.76%)   |
| Employer social insurance | €65.812bn (12.76%)  | €72.513bn (11.82%)  | €77.202bn (10.34%)  |
| Total revenue             | €217.209bn (42.12%) | €261.407bn (42.62%) | €308.681bn (41.35%) |

**Expenditure**

|                            | Period 1            | Period 20           | Period 50           |
|:---------------------------|:--------------------|:--------------------|:--------------------|
| Government consumption     | €150.161bn (29.12%) | €141.919bn (23.14%) | €193.433bn (25.91%) |
| Unemployment benefits      | €8.653bn (1.68%)    | €4.784bn (0.78%)    | €4.931bn (0.66%)    |
| Interest payments on debt  | €0.000bn (0.00%)    | €1.091bn (0.18%)    | €16.510bn (2.21%)   |
| Household social transfers | €198.168bn (38.43%) | €173.422bn (28.27%) | €201.410bn (26.98%) |
| Public pension benefits    | €133.923bn (25.97%) | €140.251bn (22.87%) | €163.410bn (21.89%) |
| Other social transfers     | €26.758bn (5.19%)   | €28.022bn (4.57%)   | €32.649bn (4.37%)   |
| Necessity support          | €28.834bn (5.59%)   | €0.365bn (0.06%)    | €0.421bn (0.06%)    |
| Total expenditure          | €348.328bn (67.55%) | €316.432bn (51.59%) | €411.354bn (55.11%) |

## Permanent-income decomposition

In [27]:
decomposition = build_permanent_income_log_ratio_decomposition_df(
    simulation,
    country_code=COUNTRY,
    reducer="mean",
    include_log_real_pc_income=True,
)
plot_permanent_income_log_ratio_decomposition(
    simulation,
    country_code=COUNTRY,
    columns=["ln_y_p_over_y", "common_log_ratio"],
    reducer="mean",
)

fig = plot_permanent_income_log_ratio_decomposition(
    simulation,
    country_code=COUNTRY,
    columns=["real_pc_income_idx"],
    reducer="mean",
    title='real_pc_income_idx',
    show=True,
)


contributions = build_permanent_income_forecast_contribution_table(
    simulation,
    country_code=COUNTRY,
    periods=list(range(9)),
    include_fixed=False,
)
contributions


,period,date,regressor,simulation_source,is_fixed,x_t,coefficient,contribution,point_forecast
0,0,2014Q1,time_trend,simulation_period_index,False,136.000000,0.004948,0.672896,0.022717
1,0,2014Q1,covid19,excluded_stage_3_dummy,False,0.000000,0.002059,0.000000,0.022717
2,0,2014Q1,log_real_pc_income,real_pc_income,False,4.605170,-1.017999,-4.688058,0.022717
3,0,2014Q1,d4_log_real_pc_income,real_pc_income,False,0.002550,0.068133,0.000174,0.022717
4,0,2014Q1,real_interest_rate_ma4_l1,policy_rate_and_cpi_fixed_basket,False,-0.757011,0.000638,-0.000483,0.022717
...,...,...,...,...,...,...,...,...,...
76,8,2016Q1,real_interest_rate_ma4_l1,policy_rate_and_cpi_fixed_basket,False,-0.974541,0.000638,-0.000622,0.062007
77,8,2016Q1,real_interest_rate_ma4_l5,policy_rate_and_cpi_fixed_basket,False,-0.487531,-0.002580,0.001258,0.062007
78,8,2016Q1,real_interest_rate_ma4_l9,policy_rate_and_cpi_fixed_basket,False,-0.996777,0.001678,-0.001673,0.062007
79,8,2016Q1,unemp_rate_ma4_l1,unemployment_rate,False,9.341991,-0.003852,-0.035985,0.062007


## Sector and firm-credit diagnostics

In [28]:
firm_sector_groups_table(model, COUNTRY)
plot_cumulative_insolvent_firms_by_sector(df_scenario)

credit_panels = [
    ["total_target_short_term_credit", "total_received_short_term_credit"],
    ["total_target_long_term_credit", "total_received_long_term_credit"],
    "short_term_loan_debt",
    "long_term_loan_debt",
]
plot_agent_timeseries(
    model,
    COUNTRY,
    "firms",
    variables=credit_panels,
    agg="sum",
    no_cols=2,
    show_legend=False,
    **FIGURE_SIZES["dense"],
)
plot_firm_credit_to_equity_and_capital(model, COUNTRY, show=True, return_df=False)
plot_sector_tfp_investment_desired_mb_mc_ratio(model, COUNTRY)


## Household Diagnostics

In [29]:
hh_vars = [
    "total_target_consumption_loans",
    [
        "target_consumption",
        "amount_bought",
        "consumption",
        "total_consumption",
    ],
    ["target_investment", "total_investment"],
    [
        "expected_income",
        "income",
    ],
    ["consumption_loan_debt", "total_consumption_loan_debt", "received_consumption_loans"],
    "liquidity_shortfall",
    "household_saving",
    "total_mortgage_debt",
    "portfolio_actual_illiquid_share",
    "portfolio_target_tfa_base",
    "portfolio_post_return_lfa",
    "portfolio_post_return_ifa",
    "portfolio_liquid_return_rate",
    "portfolio_illiquid_return_rate",
    "portfolio_investable_surplus",
    "portfolio_target_illiquid_share",
    "portfolio_target_illiquid_assets",
]


from src.visual_helpers import plot_agent_timeseries  # noqa: E402

fig = plot_agent_timeseries(
    model,
    "FRA",
    "households",
    hh_vars,
    # panel_titles=["Target vs bought"],
    agg="sum",  # or "mean"/"median"
    # no_cols=2,
    show_legend=True,
    show=False,
    base_height=300,
    base_width=600,
)

fig.show()

In [30]:
from src.visual_helpers import plot_agent_timeseries

fig = plot_agent_timeseries(
    model,
    "FRA",
    "households",
    [
        ["target_consumption", "consumption", "amount_bought"],
        [
            "wealth_real_assets",
            "wealth_main_residence",
            "wealth_other_properties",
            "wealth_other_real_assets",
            "wealth_deposits",
            "wealth_other_financial_assets",
            "wealth_financial_assets",
        ],
        ["wealth", "net_wealth", "debt"],
        [
            "expected_income",
            "expected_income_employee",
            "expected_income_social_transfers",
            "expected_income_financial_assets",
        ],
        ["income", "income_employee", "income_social_transfers", "income_rental", "income_financial_assets"],
        ["target_investment", "investment", "total_investment", "total_investment_before_vat"],
    ],
    # panel_titles=["Target vs bought"],
    agg="sum",  # or "mean"/"median"
    no_cols=2,
    show_legend=True,
    show=False,
    base_height=300,
    base_width=600,
)

fig.show()

ValueError: households.ts has no field 'wealth_deposits'.

In [ ]:
fig = plot_agent_timeseries(
    model,
    "FRA",
    "households",
    [
        ["mortgage_debt", "consumption_loan_debt", "debt"],
        ["target_mortgage", "received_mortgages", "received_consumption_loans"],
        ["debt_installments", "interest_paid_on_deposits", "interest_paid_on_loans", "interest_paid"],
        [
            "real_amount_sold",
            "real_amount_sold_to_FRA",
            "real_amount_sold_to_ROW",
            "nominal_amount_sold_in_lcu",
            "nominal_amount_sold_in_lcu_to_FRA",
            "nominal_amount_sold_in_lcu_to_ROW",
        ],
        [
            "nominal_amount_spent_in_usd",
            "nominal_amount_spent_in_usd_to_FRA",
            "nominal_amount_spent_in_usd_to_ROW",
            "nominal_amount_spent_in_lcu",
            "nominal_amount_spent_in_lcu_to_FRA",
            "nominal_amount_spent_in_lcu_to_ROW",
        ],
        ["real_amount_bought", "real_amount_bought_from_FRA", "real_amount_bought_from_ROW"],
    ],
    agg="sum",
    no_cols=2,
    show_legend=True,
    show=False,
    base_height=300,
    base_width=600,
)

fig.show()

In [ ]:
fig = plot_agent_timeseries(
    model,
    "FRA",
    "households",
    ["price_paid_for_property", ["rent", "rent_imputed"], "max_price_willing_to_pay", "max_rent_willing_to_pay"],
    # panel_titles=["Target vs bought"],
    agg="sum",  # or "mean"/"median"
    no_cols=2,
    show_legend=True,
    show=False,
    base_height=300,
    base_width=600,
)

fig.show()

In [ ]:
fig = plot_agent_timeseries(
    model,
    "FRA",
    "households",
    [
        ["mortgage_debt", "consumption_loan_debt"],
        "price_paid_for_property",
        "wealth_real_assets",
        "wealth_main_residence",
        "wealth_other_properties",
        "wealth_other_real_assets",
        "wealth_deposits",
        "wealth_other_financial_assets",
        "wealth_financial_assets",
    ],
    # panel_titles=["Target vs bought"],
    agg="sum",  # or "mean"/"median"
    no_cols=4,
    show_legend=True,
    show=False,
    base_height=300,
    base_width=400,
)

fig.show()

In [ ]:
fig = plot_agent_timeseries(
    model,
    "FRA",
    "households",
    [
        "income_rental",
        "wealth_real_assets",
        "wealth_main_residence",
        "wealth_other_properties",
        "wealth_other_real_assets",
        "wealth_deposits",
        "wealth_other_financial_assets",
        "wealth_financial_assets",
    ],
    # panel_titles=["Target vs bought"],
    agg="sum",  # or "mean"/"median"
    no_cols=4,
    show_legend=True,
    show=False,
    # condition="wealth_other_financial_assets<0",
    agent_id=["5169"],
    base_height=300,
    base_width=400,
)

fig.show()

## Optional Monte Carlo

In [ ]:
if RUN_MONTE_CARLO:
    rng = np.random.default_rng(run_config.seed)
    mc_seeds = rng.choice(np.arange(1000), size=50, replace=False).tolist()
    mc = run_seeded_monte_carlo(
        datawrapper=data,
        country_configurations=state.country_configurations,
        country_code=COUNTRY,
        seeds=mc_seeds,
        t_max=cfg.t_max,
        n_jobs=-1,
        backend="loky",
        batch_size=1,
    )
    plot_mc(mc=mc, cols=list(MACRO_COLUMNS), no_cols=4, country_code=COUNTRY)
else:
    print("Set RUN_MONTE_CARLO = True to run seeded Monte Carlo simulations.")
